# 04 — Preprocessing Pipeline Demonstration

This notebook demonstrates the `MediSensePreprocessor` pipeline for all three datasets:
1. Load raw data
2. Show before/after preprocessing
3. Verify no NaN in output
4. Show output feature names
5. Visualize scaled distributions

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import DATASETS, SEED
from src.data.loader import load_raw_dataset, get_target_column
from src.data.preprocessor import MediSensePreprocessor, build_preprocessor
from src.data.splitter import stratified_split

sns.set_theme(style='whitegrid', palette='muted')
%matplotlib inline

## Helper Function

In [ ]:
def preprocess_and_inspect(dataset_name: str):
    """Load, clean, preprocess, and inspect a dataset."""
    print(f"{'=' * 60}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'=' * 60}")
    
    # Load raw data
    df = load_raw_dataset(dataset_name)
    target_col = get_target_column(dataset_name)
    
    # Dataset-specific cleaning
    if dataset_name == 'heart':
        df['ca'] = pd.to_numeric(df['ca'], errors='coerce')
        df['thal'] = pd.to_numeric(df['thal'], errors='coerce')
        df['target'] = (df['target'] > 0).astype(int)
    elif dataset_name == 'liver':
        df['Dataset'] = df['Dataset'].map({1: 1, 2: 0})
    
    print(f"\nRaw shape: {df.shape}")
    print(f"Missing values before preprocessing:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
    
    # Split
    train_df, val_df, test_df = stratified_split(df, target_col)
    X_train = train_df.drop(columns=[target_col])
    y_train = train_df[target_col].values
    X_val = val_df.drop(columns=[target_col])
    
    print(f"\nTrain shape: {X_train.shape}")
    print(f"Val shape:   {X_val.shape}")
    
    # Fit preprocessor
    preprocessor = MediSensePreprocessor(dataset_name)
    X_train_processed = preprocessor.fit_transform(X_train, y_train)
    X_val_processed = preprocessor.transform(X_val)
    
    # Get feature names
    feature_names = preprocessor.get_feature_names_out()
    
    print(f"\nProcessed train shape: {X_train_processed.shape}")
    print(f"Output feature names ({len(feature_names)}): {feature_names}")
    
    # Verify no NaN
    n_nan_train = np.isnan(X_train_processed).sum()
    n_nan_val = np.isnan(X_val_processed).sum()
    print(f"\nNaN in processed train: {n_nan_train}")
    print(f"NaN in processed val:   {n_nan_val}")
    assert n_nan_train == 0, "Found NaN in processed training data!"
    assert n_nan_val == 0, "Found NaN in processed validation data!"
    print("No NaN values in processed output.")
    
    return X_train_processed, feature_names, dataset_name

## Process All Three Datasets

In [ ]:
results = {}
for name in ['heart', 'diabetes', 'liver']:
    X_proc, feat_names, ds_name = preprocess_and_inspect(name)
    results[ds_name] = (X_proc, feat_names)
    print()

## Visualize Scaled Distributions (After Preprocessing)

After StandardScaler, numeric features should be approximately centered at 0 with unit variance.

In [ ]:
for ds_name, (X_proc, feat_names) in results.items():
    # Only plot the numeric features (first N before one-hot columns)
    num_feats = DATASETS[ds_name]['numeric_features']
    n_numeric = len(num_feats)
    
    n_cols = min(4, n_numeric)
    n_rows = (n_numeric + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 4 * n_rows))
    if n_numeric == 1:
        axes = [axes]
    else:
        axes = axes.flatten()
    
    for i in range(n_numeric):
        ax = axes[i]
        sns.histplot(X_proc[:, i], kde=True, ax=ax, bins=30)
        ax.set_title(f'{feat_names[i]} (scaled)')
        ax.axvline(0, color='red', linestyle='--', alpha=0.5)
    
    for j in range(n_numeric, len(axes)):
        axes[j].set_visible(False)
    
    plt.suptitle(f'Scaled Numeric Feature Distributions — {ds_name.upper()}', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

## Before vs After Comparison (Heart Dataset)

In [ ]:
# Show before/after for heart dataset
df_heart = load_raw_dataset('heart')
df_heart['ca'] = pd.to_numeric(df_heart['ca'], errors='coerce')
df_heart['thal'] = pd.to_numeric(df_heart['thal'], errors='coerce')
df_heart['target'] = (df_heart['target'] > 0).astype(int)

num_feats = DATASETS['heart']['numeric_features']

fig, axes = plt.subplots(len(num_feats), 2, figsize=(12, 4 * len(num_feats)))

X_proc_heart = results['heart'][0]
feat_names_heart = results['heart'][1]

for i, col in enumerate(num_feats):
    # Before
    axes[i, 0].hist(df_heart[col].dropna(), bins=30, color='steelblue', alpha=0.7)
    axes[i, 0].set_title(f'{col} — Raw')
    # After
    axes[i, 1].hist(X_proc_heart[:, i], bins=30, color='coral', alpha=0.7)
    axes[i, 1].set_title(f'{feat_names_heart[i]} — Preprocessed')

plt.suptitle('Before vs After Preprocessing — Heart Dataset', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## Summary

The `MediSensePreprocessor` pipeline successfully:
- Imputes missing values (median for numeric, most frequent for categorical)
- Clips outliers using IQR-based clipping
- Scales numeric features to zero mean and unit variance
- One-hot encodes categorical features (drop-first strategy)
- Produces clean output with no NaN values